In [ ]:
import holoviews as hv
import xarray as xr
import pint_xarray
import cf_xarray
import hvplot.xarray
import matplotlib.pylab as plt
from matplotlib.colors import Normalize

import numpy as np
import panel as pn

import cmcrameri.cm  # noqa: F401  pylint: disable=unused-import
import cmglaciology.cm  # noqa: F401  pylint: disable=unused-import
import cmcrameri.cm as cmc

from pism_terra.plotting import blended_animation
from pism_terra.processing import preprocess_netcdf

In [ ]:
odir="2026_10_jib_calib_2"

ds = []
for m in ["mean", "max", "sum"]:
    _ds = xr.open_mfdataset(f"/Users/andy/base/pism-terra/{odir}/output/processed_scalar/{m}_JIB_SHELF_g450m_id_OCX_uq_*_ocx_free_1985-01-01_2005-01-01.nc",
                           preprocess=preprocess_netcdf)
    _ds = _ds.expand_dims({"agg_id": [m]})
    ds.append(_ds)
ds = xr.concat(ds, dim="agg_id").set_index({"glacier_id":"glacier_id_name"}).sel({"glacier_id": "JIB_SHELF"})
ds = ds.drop_vars("vonmises_stress").pint.quantify().pint.to({"eigen1": "yr^-1",
                                                              "ice_mass_transport_across_grounding_line": "Gt/yr",
                                                              "plume_basal_melt_rate": "m/yr",
                                                              "plume_fresh_water_melt_rate": "m/yr",
                                                              "tendency_of_ice_mass_due_to_discharge": "Gt/yr"})
eigen1 = ds["eigen1"].sel({"agg_id": "mean"}).squeeze().drop_vars("agg_id")
frac = ds["fracture_density"].sel({"agg_id": "sum"}).squeeze().drop_vars("agg_id")
glf = ds["ice_mass_transport_across_grounding_line"].sel({"agg_id": "sum"}).squeeze().drop_vars("agg_id")
basal_melt = ds["plume_basal_melt_rate"].sel({"agg_id": "mean"}).squeeze().drop_vars("agg_id")
discharge = ds["tendency_of_ice_mass_due_to_discharge"].sel({"agg_id": "sum"}).squeeze().drop_vars("agg_id")
plume_discharge = ds["plume_fresh_water_melt_rate"].sel({"agg_id": "sum"}).squeeze().drop_vars("agg_id")
speed = ds["velsurf_mag"].sel({"agg_id": "mean"}).squeeze().drop_vars("agg_id")
jib = xr.merge([eigen1, frac, glf, basal_melt, discharge, plume_discharge, speed])


In [ ]:

fields = jib.drop_vars(["pism_config"], errors="ignore")
fields = fields[[v for v in fields.data_vars if fields[v].dtype.kind == "f"]]  # floats only

lo, hi = fields.min(), fields.max()          # add a dim, e.g. .min("time"), to scale per cell
span = (hi - lo).where(hi > lo)              # constant variables become NaN instead of inf
scaled = 2 * (fields - lo) / span - 1
scaled = scaled.sel(time=slice("1992", "2002"))

In [ ]:
import matplotlib.pylab as plt
import numpy as np

In [ ]:
fig, ax = plt.subplots(1, 1, sharex=True, figsize=(12, 8))
vs = ["velsurf_mag", "tendency_of_ice_mass_due_to_discharge", "plume_basal_melt_rate"]
ls = ["speed", "D", "m"]
cs = cmc.managua(np.linspace(0.1, 0.8, len(vs)))
for _v, _label, _color in zip(vs, ls, cs):
    scaled[_v].plot(ax=ax, hue="uq_id", lw=0.75, color=_color, ls="dashed")
    scaled[_v].rolling(time=7, center=True).mean().plot(ax=ax, hue="uq_id", lw=2.5, label=_label, color=_color)
ax.axhline(ls="dotted", lw=0.5)
ax.legend()
ax.set_title("Normalized")
ax.set_ylabel(None)

In [ ]:
vs = ["eigen1", "fracture_density", "ice_mass_transport_across_grounding_line", "plume_basal_melt_rate", "plume_fresh_water_melt_rate", "tendency_of_ice_mass_due_to_discharge"]
ls = ["eigen", "frac", "GLF", "m", "p", "D"]
cs = cmc.managua(np.linspace(0.1, 0.8, len(vs)))
fig, axs = plt.subplots(len(vs), 1, sharex=True, figsize=(12, 20))
for k, (_v, _label, _color) in enumerate(zip(vs, ls, cs)):
    jib[_v].plot(ax=axs[k], hue="uq_id", lw=0.75, color=_color, ls="dashed", add_legend=False)
    jib[_v].rolling(time=7, center=True).mean().plot(ax=axs[k], hue="uq_id", lw=2.5, label=_label, color=_color, add_legend=False)
    axs[k].axhline(ls="dotted", lw=0.5)
    #axs[k].legend()
    axs[k].set_xlabel(None)
    #axs[k].set_ylabel(None)
    axs[k].set_title(_label)
    
axs[-1].set_xlabel("Year")


In [ ]:
1-2e-2

In [ ]:
1.4e5-4e4

In [ ]:
import geopandas as gpd

In [ ]:
ds.ice_mass_transport_across_grounding_line.sel(agg_id="mean").plot(hue="uq_id")

In [ ]:
spatial = xr.open_mfdataset("/Users/andy/base/pism-terra/2026_10_jib_calib_2/output/processed_spatial/spatial_GIS_JIB_g450m_id_OCX_uq_*_ocx_free_1985-01-01_2005-01-01.nc", preprocess=preprocess_netcdf, parallel=True)


In [ ]:
spatial_jib = spatial.sel({"time": slice("1992", "2005")}).sel(exp_id="OCX", uq_id="7")
blended_animation(spatial_jib.velsurf_mag, spatial_jib.usurf, cmap="cmg.speed", clim=(0, 10000))
# blended_animation(spatial_jib.plume_basal_melt_rate, spatial_jib.usurf, cmap="cmc.davos_r", clim=(0, 500))
# blended_animation(spatial_jib.thk, spatial_jib.usurf, cmap="cmc.lipari_r", clim=(0, 500))

In [ ]:
blended_animation(spatial_jib.plume_basal_melt_rate, spatial_jib.usurf, cmap="cmc.davos_r", clim=(0, 500))

In [ ]:
xr.apply_ufunc?